In [13]:
import cvxpy as cp
import cobra as cb
from cobra.util.array import create_stoichiometric_matrix
from cvxpylayers.torch import CvxpyLayer
import torch

# Load the model
model = cb.io.load_json_model("../Cobra_Models/Univ_GPR_Model.json")

# Extract the stoichiometric matrix and the bounds
S = create_stoichiometric_matrix(model, array_type="dok")
lower_bounds = model.reactions.list_attr("lower_bound")
upper_bounds = model.reactions.list_attr("upper_bound")

n_rxns = S.shape[1]
n_mets = S.shape[0]

# Define the optimization variables
v = cp.Variable(n_rxns)

# add params for lower and upper bounds
lb = cp.Parameter(n_rxns)
ub = cp.Parameter(n_rxns)
lb.value = lower_bounds
ub.value = upper_bounds

# Define the constraints
constraints = [S @ v == 0]  # Steady-state constraint
for i in range(n_rxns):
    constraints.append(v[i] >= lb[i])
    constraints.append(v[i] <= ub[i])

# Define the objective function (maximize the flux through the biomass reaction)
biomass_index = model.reactions.index("bio1(c)")
objective = cp.Maximize(v[biomass_index])

# Define the optimization problem
problem = cp.Problem(objective, constraints)

# Create a CVXPY layer
layer = CvxpyLayer(problem, parameters=[lb, ub], variables=[v])

# Example input for the layer (using the original bounds)
input_lb = torch.tensor(lower_bounds)
input_ub = torch.tensor(upper_bounds)

# Forward pass through the layer
solution, = layer(input_lb, input_ub)

# mapped back to reaction names
reaction_names = [rxn.id for rxn in model.reactions]
flux_dict = {reaction_names[i]: solution[i].item() for i in range(n_rxns)}
for rxn, flux in flux_dict.items():
    print(f"{rxn}: {flux}")


EX_A(e): -320.39299740576314
EX_B(e): -382.2501043983634
EX_C(e): -329.5429524192797
EX_D(e): -368.2185857120407
EX_E(e): -350.25003713255154
T_A: 320.39299796511295
T_B: -382.25010483394993
T_C: 329.5429529276575
T_D: 368.21858607140155
T_E: 350.2500376292343
T_Bio: 999.9980323311813
AtoPrebio(c): 157.03081493749974
BtoPrebio(c): 67.79293645269958
CtoPrebio(c): 157.03081493749974
DtoPrebio(c): 157.03081493749974
EtoPrebio(c): 157.03081493749974
AtoB(c): -174.50891749107052
AtoC(c): -279.62162287965583
AtoD(c): -202.57491898533007
AtoE(c): -238.30770655457994
BtoC(c): -105.11270538848376
BtoD(c): -28.066001494546043
BtoE(c): -63.798789063408854
CtoD(c): 77.04670389422164
CtoE(c): 41.31391632507494
DtoE(c): -35.732787568942655
ABtoC(c): 336.9257484682789
ADtoE(c): 109.64908218493453
CEtoPrebio(c): 163.3737548128281
ADtoPrebio(c): 140.70807621588202
bio1(c): 999.9980310565404
EX_Bio(e): 999.9980329775038


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

class FBA_Net(nn.Module):
    def __init__(self, cvxpylayer, n_input, n_bounds, n_hidden_layers, hidden_layer_size):
        super(FBA_Net, self).__init__()
        self.cvxpylayer = cvxpylayer
        
        # --- ICNN Architecture Components ---
        # 1. Initial layer: Maps input (media) to first hidden state
        self.first_layer = nn.Linear(n_input, hidden_layer_size)
        
        # 2. Hidden Layers: 
        # z_layers: Must have non-negative weights (Wz)
        # x_layers: Skip connections from media input (Wx)
        self.z_layers = nn.ModuleList([
            nn.Linear(hidden_layer_size, hidden_layer_size, bias=False) 
            for _ in range(n_hidden_layers - 1)
        ])
        self.x_layers = nn.ModuleList([
            nn.Linear(n_input, hidden_layer_size, bias=True) 
            for _ in range(n_hidden_layers - 1)
        ])
        
        # 3. Output Layer: Maps to n_bounds (e.g., lower/upper bound vectors)
        # We split these to ensure the bound generation stays convex
        self.final_z = nn.Linear(hidden_layer_size, n_bounds, bias=False)
        self.final_x = nn.Linear(n_input, n_bounds, bias=True)

    def forward(self, media_input):
        # --- ICNN Forward Pass ---
        # Propagate through the first layer
        z = F.relu(self.first_layer(media_input))
        
        # Propagate through hidden layers with skip connections
        for z_layer, x_layer in zip(self.z_layers, self.x_layers):
            z = F.relu(z_layer(z) + x_layer(media_input))
        
        # Generate the learned bounds (h)
        # These will be the parameters fed into your cvxpylayer
        learned_h = self.final_z(z) + self.final_x(media_input)
        
        # --- CVXPY Layer Pass ---
        # Note: Adjust parameters based on your specific LP setup (e.g., Gz <= h)
        # If your LP requires both lb and ub, you can split learned_h accordingly.
        solution, = self.cvxpylayer(learned_h)
        
        return solution, learned_h

    def enforce_convexity(self):
        """Call this after optimizer.step() to keep the ICNN weights non-negative."""
        with torch.no_grad():
            for layer in self.z_layers:
                layer.weight.clamp_(min=0)
            self.final_z.weight.clamp_(min=0)